# **P714 Inventory Management**

Exploratory Data Analysis, Preprocessing, Model Building and Evaluation

**Project objective:** Explore the inventory data and predict inventory risks whether a product is at risk of stockout or overstock.

**Data Set Details:**

•	**Name:** Stock Risk Prediction Dataset

•	**Size:** 5,000 rows × 11 columns

•	**Type:** Tabular data

•	**Format:** Excel (.xlsx)

•	**Target Column:** stock_status (binary: stockout, overstock)


**Stock-Status Rule**

In all 5,000 supplied records, stock_status is stockout when current_stock < reorder_point, and overstock when current_stock > reorder_point. There are no equal values. Since both rule inputs are model features, scores near 99% are expected: the models are largely learning an existing stock rule, rather than demonstrating future stockout forecasting.

**Part 1: Exploratory Data Analysis**

1. Import Libraries

In [ ]:
# Import the required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


2. Load the Dataset

In [ ]:
# Load and display the dataset
df=pd.read_excel('/content/stock_risk_prediction_dataset (1).xlsx')
df.head()


3. Check Shape

In [ ]:
# Check the number of rows and columns
df.shape


The dataset contains 5,000 rows and 11 columns

4. Check Dataset Information

In [ ]:
# Check the dataset information
df.info()


5. Check Missing Values

In [ ]:
# Check for null values
df.isnull().sum()


6. Check Duplicate Values

In [ ]:
# Check for duplicate values
df.duplicated().sum()


No missing values or duplicate records were found.

7. Check Descriptive Statistics

In [ ]:
# Describe the numerical columns
df.describe()


Descriptive statistics summarise the centre, spread and range of numerical features.

8. Check Target Variable Distribution

In [ ]:
# Check the target class distribution
df['stock_status'].value_counts()


The target is imbalanced: 4,543 overstock records (90.86%) and 457 stockout records (9.14%). Stockout recall and F1-score are therefore evaluated alongside accuracy.

**Part 2: EDA visualizations**

1. Boxplot

In [ ]:
# Check for outliers in the numerical columns
num_col = df.select_dtypes(include=['int64', 'float64']).columns

for i in num_col:
    sns.boxplot(df[i])
    plt.title(i)
    plt.show()


Boxplots identify extreme numerical values.

Outlier capping is not applied because extreme values may represent meaningful inventory conditions, such as excess stock or unusually high demand.

2. Histogram with KDE Plot

In [ ]:
# Plot the distribution of numerical values
num_col = df.select_dtypes(include=['int64', 'float64']).columns
for i in num_col:
    sns.histplot(data=df, x=i, kde=True)
    plt.title(i)
    plt.show()


Histograms show the distribution of numerical features.

3. Countplot

In [ ]:
# Analyse the categorical values
cat_col=df.select_dtypes(include=['object']).columns
for i in cat_col:
    sns.countplot(data=df,x=i)
    plt.title(i)
    plt.show()


Countplots show the frequencies of categorical values.

**Part 3: Data Pre-processing**

1. Separate Feature and Target

In [ ]:
# Separate the input features and target column
x=df.drop(columns='stock_status')
y=df['stock_status']


x contains the input features and y contains the target, stock_status.

2. Train-Test Split

In [ ]:
# Split the dataset into training and testing data
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=100,stratify=y)

print("X_train:", x_train.shape)
print("X_test:", x_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


3. Encode Categorical Features

Fit encoders on training data, then transform testing data.

In [ ]:
# Convert categorical values into numerical values
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
oe = OrdinalEncoder()
le = LabelEncoder()
cat_col = ['season', 'item_category', 'supplier_reliability']
x_train[cat_col] = oe.fit_transform(x_train[cat_col])
x_test[cat_col] = oe.transform(x_test[cat_col])
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)


The split contains 80% training data and 20% testing data. Stratification preserves the class proportions.

4. Standard Scaler

In [ ]:
# Scale the training and testing data
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train_scaled = sc.fit_transform(x_train)
x_test_scaled = sc.transform(x_test)


StandardScaler is fitted on training data and applied to testing data. Scaled features are used for Logistic Regression and SVC; tree models use unscaled encoded features.

5. Correlation Heatmap

The heatmap shows relationships within the encoded training data.

In [ ]:
# Plot the correlation heatmap using training data
train_corr = x_train.copy()
train_corr['stock_status'] = y_train
plt.figure(figsize=(10,10))
sns.heatmap(train_corr.corr(numeric_only=True), annot=True)
plt.title('Training Data Correlation')
plt.show()


**Most Frequent Class Baseline**

Always predicting the most frequent training class gives a reference accuracy. This baseline does not detect any stockouts.

In [ ]:
# Measure accuracy from always predicting the most frequent training class
print('Most frequent class baseline test accuracy:',np.mean(y_test == np.bincount(y_train).argmax()))


**Part 4: Model Building**

Train four classifiers. Logistic Regression and SVC use scaled features; tree models use encoded, unscaled features. All models use class_weight='balanced' to give more influence to the minority stockout class during training.

1. Logistic Regression

In [ ]:
# Train the model
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression(class_weight='balanced',random_state=42)
lr.fit(x_train_scaled,y_train)

2. Decision Tree

In [ ]:
# Train the model
from sklearn.tree import DecisionTreeClassifier
DT=DecisionTreeClassifier(random_state=42,class_weight='balanced')
DT.fit(x_train,y_train)

3. Random Forest

In [ ]:
# Train the model
from sklearn.ensemble import RandomForestClassifier
rf_model=RandomForestClassifier(n_estimators=100,random_state=42,max_samples=0.6,max_features=0.7,max_depth=10,class_weight='balanced')
rf_model.fit(x_train,y_train)

4. SVC

In [ ]:
# Train the model
from sklearn.svm import SVC
svc=SVC(kernel='linear',class_weight='balanced',random_state=42)
svc.fit(x_train_scaled,y_train)

**Part 5: Hyperparameter Tuning**

All four models are tuned using five-fold cross-validation on training data, with **stockout F1-score** (`scoring='f1'`) as the selection metric. Balanced class weights are retained. Logistic Regression and SVC use pipelines so that scaling is fitted only within each cross-validation training fold. The test set is not used for choosing hyperparameters.

1. Logistic Regression Tuning

Compare regularisation strengths and solvers. The pipeline applies scaling inside each training fold.

In [ ]:
# Tune Logistic Regression using a pipeline
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
lr_pipeline=Pipeline([
    ('scaler',StandardScaler()),
    ('lr',LogisticRegression(class_weight='balanced',max_iter=1000,random_state=42))
])
lr_params={
    'lr__C':[0.01,0.1,1,10,100],
    'lr__solver':['lbfgs','liblinear']
}
lr_grid_search=GridSearchCV(lr_pipeline,lr_params,cv=5,scoring='f1',n_jobs=-1)
lr_grid_search.fit(x_train,y_train)
best_lr_model=lr_grid_search.best_estimator_
print('Best parameters:',lr_grid_search.best_params_)
print('Best CV stockout F1:',lr_grid_search.best_score_)


2. Decision Tree Tuning

In [ ]:
# Tune the Decision Tree model
from sklearn.model_selection import GridSearchCV

dt_model=DecisionTreeClassifier(random_state=42,class_weight='balanced')

params={'criterion':['gini','entropy'],'splitter':['best','random'],'max_depth':range(1,10)}

dt_grid_search=GridSearchCV(estimator=dt_model,param_grid=params,cv=5,scoring='f1',n_jobs=-1)

dt_grid_search.fit(x_train,y_train)
print('Best parameters:',dt_grid_search.best_params_)
print('Best CV stockout F1:',dt_grid_search.best_score_)
best_dt_model=dt_grid_search.best_estimator_

3. Random Forest Tuning

In [ ]:
# Tune the Random Forest model
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_samples': [0.6, 0.7, 0.8],
    'max_features': [0.6, 0.7, 0.8],
    'max_depth': [4, 6, 8, 10]
}
rf_search_model = RandomForestClassifier(random_state=42,class_weight='balanced')
rf_grid_search = GridSearchCV(estimator=rf_search_model, param_grid=param_grid, cv=5, scoring='f1', n_jobs=-1)
rf_grid_search.fit(x_train, y_train)
print("Best hyperparameters:", rf_grid_search.best_params_)
print("Best CV stockout F1:", rf_grid_search.best_score_)
best_rf_model=rf_grid_search.best_estimator_


4. SVC Tuning

Try linear and RBF kernels with different regularisation values. The pipeline scales the training data within each cross-validation fold.

In [ ]:
# Tune SVC using a pipeline
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
svc_pipeline=Pipeline([
    ('scaler',StandardScaler()),
    ('svc',SVC(class_weight='balanced',random_state=42))
])
svc_params={
    'svc__C':[0.1,1,10],
    'svc__kernel':['linear','rbf']
}
svc_grid_search=GridSearchCV(svc_pipeline,svc_params,cv=5,scoring='f1',n_jobs=-1)
svc_grid_search.fit(x_train,y_train)
best_svc_model=svc_grid_search.best_estimator_
print('Best parameters:',svc_grid_search.best_params_)
print('Best CV stockout F1:',svc_grid_search.best_score_)

**Part 6: Model Evaluation**

For each original and tuned model, compare training and testing accuracy, followed by both classification reports and a test confusion matrix. Label 0 means overstock; label 1 means stockout. A final side-by-side comparison includes all four tuned models.

Logistic Regression Evaluation

In [ ]:
# Evaluate the Logistic Regression model
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
pred_train_lr=lr.predict(x_train_scaled)
pred_test_lr=lr.predict(x_test_scaled)
print('Training accuracy:',accuracy_score(y_train,pred_train_lr))
print('Testing accuracy:',accuracy_score(y_test,pred_test_lr))
print('Training classification report:')
print(classification_report(y_train,pred_train_lr,digits=4,zero_division=0))
print('Testing classification report:')
print(classification_report(y_test,pred_test_lr,digits=4,zero_division=0))
sns.heatmap(confusion_matrix(y_test,pred_test_lr),annot=True,fmt='d')
plt.title('Logistic Regression - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


Decision Tree Evaluation

In [ ]:
# Evaluate the Decision Tree model
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
pred_train_dt=DT.predict(x_train)
pred_test_dt=DT.predict(x_test)
print('Training accuracy:',accuracy_score(y_train,pred_train_dt))
print('Testing accuracy:',accuracy_score(y_test,pred_test_dt))
print('Training classification report:')
print(classification_report(y_train,pred_train_dt,digits=4,zero_division=0))
print('Testing classification report:')
print(classification_report(y_test,pred_test_dt,digits=4,zero_division=0))
sns.heatmap(confusion_matrix(y_test,pred_test_dt),annot=True,fmt='d')
plt.title('Decision Tree - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


Random Forest Evaluation

In [ ]:
# Evaluate the Random Forest model
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
pred_train_rf=rf_model.predict(x_train)
pred_test_rf=rf_model.predict(x_test)
print('Training accuracy:',accuracy_score(y_train,pred_train_rf))
print('Testing accuracy:',accuracy_score(y_test,pred_test_rf))
print('Training classification report:')
print(classification_report(y_train,pred_train_rf,digits=4,zero_division=0))
print('Testing classification report:')
print(classification_report(y_test,pred_test_rf,digits=4,zero_division=0))
sns.heatmap(confusion_matrix(y_test,pred_test_rf),annot=True,fmt='d')
plt.title('Random Forest - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


SVC Evaluation

In [ ]:
# Evaluate the SVC model
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
pred_train_svc=svc.predict(x_train_scaled)
pred_test_svc=svc.predict(x_test_scaled)
print('Training accuracy:',accuracy_score(y_train,pred_train_svc))
print('Testing accuracy:',accuracy_score(y_test,pred_test_svc))
print('Training classification report:')
print(classification_report(y_train,pred_train_svc,digits=4,zero_division=0))
print('Testing classification report:')
print(classification_report(y_test,pred_test_svc,digits=4,zero_division=0))
sns.heatmap(confusion_matrix(y_test,pred_test_svc),annot=True,fmt='d')
plt.title('SVC - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


Tuned Logistic Regression Evaluation

In [ ]:
# Evaluate the tuned Logistic Regression model
pred_train_lr_best=best_lr_model.predict(x_train)
pred_test_lr_best=best_lr_model.predict(x_test)
print('Training accuracy:',accuracy_score(y_train,pred_train_lr_best))
print('Testing accuracy:',accuracy_score(y_test,pred_test_lr_best))
print('Training classification report:')
print(classification_report(y_train,pred_train_lr_best,digits=4,zero_division=0))
print('Testing classification report:')
print(classification_report(y_test,pred_test_lr_best,digits=4,zero_division=0))
sns.heatmap(confusion_matrix(y_test,pred_test_lr_best),annot=True,fmt='d')
plt.title('Tuned Logistic Regression - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


Tuned Decision Tree Evaluation

In [ ]:
# Evaluate the Tuned Decision Tree model
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
pred_train_dt_best=best_dt_model.predict(x_train)
pred_test_dt_best=best_dt_model.predict(x_test)
print('Training accuracy:',accuracy_score(y_train,pred_train_dt_best))
print('Testing accuracy:',accuracy_score(y_test,pred_test_dt_best))
print('Training classification report:')
print(classification_report(y_train,pred_train_dt_best,digits=4,zero_division=0))
print('Testing classification report:')
print(classification_report(y_test,pred_test_dt_best,digits=4,zero_division=0))
sns.heatmap(confusion_matrix(y_test,pred_test_dt_best),annot=True,fmt='d')
plt.title('Tuned Decision Tree - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


Tuned Random Forest Evaluation

In [ ]:
# Evaluate the Tuned Random Forest model
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
pred_train_rf_best=best_rf_model.predict(x_train)
pred_test_rf_best=best_rf_model.predict(x_test)
print('Training accuracy:',accuracy_score(y_train,pred_train_rf_best))
print('Testing accuracy:',accuracy_score(y_test,pred_test_rf_best))
print('Training classification report:')
print(classification_report(y_train,pred_train_rf_best,digits=4,zero_division=0))
print('Testing classification report:')
print(classification_report(y_test,pred_test_rf_best,digits=4,zero_division=0))
sns.heatmap(confusion_matrix(y_test,pred_test_rf_best),annot=True,fmt='d')
plt.title('Tuned Random Forest - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


Tuned SVC Evaluation

In [ ]:
# Evaluate the tuned SVC model
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
pred_train_svc_best=best_svc_model.predict(x_train)
pred_test_svc_best=best_svc_model.predict(x_test)
print('Training accuracy:',accuracy_score(y_train,pred_train_svc_best))
print('Testing accuracy:',accuracy_score(y_test,pred_test_svc_best))
print('Training classification report:')
print(classification_report(y_train,pred_train_svc_best,digits=4,zero_division=0))
print('Testing classification report:')
print(classification_report(y_test,pred_test_svc_best,digits=4,zero_division=0))
sns.heatmap(confusion_matrix(y_test,pred_test_svc_best),annot=True,fmt='d')
plt.title('Tuned SVC - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

Comparison of All Four Tuned Models

Compare test-set scores for the positive class (stockout = 1). Hyperparameters were selected using training-set cross-validation; the test set is used here only for final evaluation.

In [ ]:
# Compare all four tuned models on the same test set
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
results={
    'Tuned Logistic Regression':pred_test_lr_best,
    'Tuned Decision Tree':pred_test_dt_best,
    'Tuned Random Forest':pred_test_rf_best,
    'Tuned SVC':pred_test_svc_best
}
for name,pred in results.items():
    print(name)
    print('Testing accuracy:',round(accuracy_score(y_test,pred),4))
    print('Stockout precision:',round(precision_score(y_test,pred),4))
    print('Stockout recall:',round(recall_score(y_test,pred),4))
    print('Stockout F1:',round(f1_score(y_test,pred),4))
    print()

**Part 7: Tree Plots**

Display depth is limited to 3 for readability. Predictions still use the complete model.

Decision Tree

In [ ]:
# Plot the tree
from sklearn.tree import plot_tree
plt.figure(figsize=(22,10))
plot_tree(DT,filled=True,feature_names=x_train.columns,class_names=['overstock','stockout'],max_depth=3,rounded=True,fontsize=9)
plt.title('Decision Tree')
plt.show()


Tuned Decision Tree

In [ ]:
# Plot the tree
from sklearn.tree import plot_tree
plt.figure(figsize=(22,10))
plot_tree(best_dt_model,filled=True,feature_names=x_train.columns,class_names=['overstock','stockout'],max_depth=3,rounded=True,fontsize=9)
plt.title('Tuned Decision Tree')
plt.show()


First Tree in Tuned Random Forest

In [ ]:
# Plot the tree
from sklearn.tree import plot_tree
plt.figure(figsize=(22,10))
plot_tree(best_rf_model.estimators_[0],filled=True,feature_names=x_train.columns,class_names=['overstock','stockout'],max_depth=3,rounded=True,fontsize=9)
plt.title('First Tree in Tuned Random Forest')
plt.show()


**Part 8: Logistic Regression ROC Curve and AUC**

This follows the ROC pattern in the Logistic Regression class notebooks. Stockout is the positive class.

In [ ]:
# Calculate Logistic Regression probabilities and AUC
from sklearn.metrics import roc_auc_score,roc_curve
sigmoid=lr.predict_proba(x_test_scaled)[:,1]
auc_score=roc_auc_score(y_test,sigmoid)
print('ROC-AUC:',auc_score)


In [ ]:
# Plot the ROC curve
fpr,tpr,thr=roc_curve(y_test,sigmoid)
plt.plot(fpr,tpr,label=f'AUC={auc_score:.4f}')
plt.plot([0,1],[0,1])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Logistic Regression ROC Curve')
plt.legend()
plt.show()


**Conclusion**

All four models have now been tuned with the same five-fold cross-validation metric (stockout F1). On this test split, tuned Logistic Regression gives 99.60% accuracy, 100% stockout recall and 97.85% stockout F1. Tuned Decision Tree also gives 99.60% accuracy but misses two of the 91 stockouts; tuned SVC gives 99.50% accuracy and 100% stockout recall, but produces five false stockout alerts compared with four for tuned Logistic Regression. We prefer tuned Logistic Regression **on this split because identifying all stockouts is a priority and its false-alert count is lower than tuned SVC**. These differences are small and may change on other samples.

The labels exactly follow current_stock versus reorder_point, so high scores mainly show recovery of that rule. The data appears artificial or rule-generated; its origin is not independently confirmed. A direct rule is an appropriate benchmark. Independent real inventory data is needed before claiming future stockout prediction or deployment readiness.